# Day 3：手写 LoRA 实现 — 从 LoRALinear 到 LLaMA-LoRA

> **目标**：从零实现 LoRA 的核心模块——LoRALinear 层；将 LoRA 注入 Week4 手写的 LLaMA 模型架构中；实现冻结/训练参数管理、权重合并与卸载；在一个小规模 SFT 任务上验证 LoRA 微调的正确性，对比 Full FT 的效果和参数效率。

---

## 总体流程

```
Part 1: 手写 LoRALinear 层
  LoRA 前向传播 → 初始化策略 → Dropout → 缩放因子

Part 2: LoRA 注入 LLaMA
  自动替换 Linear 层 → 冻结预训练权重 → 可训练参数统计

Part 3: 权重合并与管理
  merge / unmerge → 导出 LoRA 权重 → 加载 LoRA 权重

Part 4: LoRA SFT 训练验证
  数据准备 → LoRA 训练循环 → 生成评估 → 与 Full FT 对比

Part 5: 参数效率分析
  参数量 → 显存 → 训练速度对比
```

In [1]:
import math
import json
import copy
import time
from dataclasses import dataclass
from typing import Optional, List, Dict, Tuple, Set

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")

Device: cuda
GPU: NVIDIA GeForce RTX 4090


## Part 1：手写 LoRALinear 层

### 1.1 核心公式回顾

$$h = W_0 x + \frac{\alpha}{r} \cdot B A x$$

- $W_0 \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$：冻结的预训练权重
- $B \in \mathbb{R}^{d_{\text{out}} \times r}$：可训练，初始化为零
- $A \in \mathbb{R}^{r \times d_{\text{in}}}$：可训练，Kaiming 初始化
- $r$：低秩分解的秩
- $\alpha$：缩放常数

In [2]:
class LoRALinear(nn.Module):
    """
    LoRA 适配的 Linear 层。
    
    h = W₀x + (α/r) · Dropout(BAx)
    
    W₀ 冻结，只训练 A 和 B。
    训练完成后可将 BA 合并到 W₀ 中，推理零开销。
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        r: int = 8,
        lora_alpha: int = 16,
        lora_dropout: float = 0.0,
        bias: bool = False,
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.r = r
        self.lora_alpha = lora_alpha
        self.scaling = lora_alpha / r
        self.merged = False

        # 原始 Linear 层（冻结）
        self.linear = nn.Linear(in_features, out_features, bias=bias)
        self.linear.weight.requires_grad = False
        if bias and self.linear.bias is not None:
            self.linear.bias.requires_grad = False

        # LoRA 低秩矩阵
        # A: r × in_features (降维)
        # B: out_features × r (升维)
        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))

        # Dropout
        self.lora_dropout = nn.Dropout(p=lora_dropout) if lora_dropout > 0 else nn.Identity()

        # 初始化
        self.reset_lora_parameters()

    def reset_lora_parameters(self):
        """A 用 Kaiming 均匀初始化，B 初始化为零。"""
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 如果已合并，直接用合并后的权重
        if self.merged:
            return self.linear(x)

        # h = W₀x + (α/r) · B(A(Dropout(x)))
        base_out = self.linear(x)
        lora_out = self.lora_dropout(x) @ self.lora_A.T @ self.lora_B.T
        return base_out + self.scaling * lora_out

    def merge(self):
        """将 LoRA 权重合并到基座权重中（推理时使用）。"""
        if not self.merged:
            # W' = W₀ + (α/r) · BA
            self.linear.weight.data += self.scaling * (self.lora_B @ self.lora_A)
            self.merged = True

    def unmerge(self):
        """从基座权重中移除 LoRA 权重（恢复训练或切换 LoRA 时使用）。"""
        if self.merged:
            self.linear.weight.data -= self.scaling * (self.lora_B @ self.lora_A)
            self.merged = False

    def extra_repr(self):
        return (
            f"in_features={self.in_features}, out_features={self.out_features}, "
            f"r={self.r}, alpha={self.lora_alpha}, scaling={self.scaling:.2f}, "
            f"merged={self.merged}"
        )

### 1.2 验证 LoRALinear

测试几个关键属性：
1. 初始时 LoRA 输出为零（$B=0$）
2. 合并前后输出一致
3. 只有 LoRA 参数可训练

In [3]:
# 测试 1: 初始时 LoRA 输出为零
lora_layer = LoRALinear(in_features=64, out_features=128, r=8, lora_alpha=16)
x = torch.randn(2, 10, 64)

# 初始时 LoRA 分支输出应该为零，总输出应等于 W₀x
with torch.no_grad():
    lora_output = lora_layer(x)
    base_output = lora_layer.linear(x)
    diff = (lora_output - base_output).abs().max().item()
    print(f"Test 1 - 初始时 LoRA 输出 vs 基座输出最大差异: {diff:.2e}")
    assert diff < 1e-6, "初始时 LoRA 输出应等于基座输出！"
    print("  ✅ 通过：B=0 保证了初始输出等于预训练模型")

print()

# 测试 2: 模拟训练（随机更新 B），验证 merge/unmerge
with torch.no_grad():
    lora_layer.lora_B.data = torch.randn_like(lora_layer.lora_B) * 0.1

with torch.no_grad():
    before_merge = lora_layer(x).clone()
    lora_layer.merge()
    after_merge = lora_layer(x).clone()
    diff = (before_merge - after_merge).abs().max().item()
    print(f"Test 2 - merge 前后输出最大差异: {diff:.2e}")
    assert diff < 1e-5, "merge 前后输出应一致！"
    print("  ✅ 通过：权重合并后输出不变")

    lora_layer.unmerge()
    after_unmerge = lora_layer(x).clone()
    diff = (before_merge - after_unmerge).abs().max().item()
    print(f"Test 2 - unmerge 后输出最大差异: {diff:.2e}")
    assert diff < 1e-5, "unmerge 后输出应恢复！"
    print("  ✅ 通过：unmerge 后输出恢复")

print()

# 测试 3: 检查可训练参数
trainable = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in lora_layer.parameters() if not p.requires_grad)
total = trainable + frozen
print(f"Test 3 - 参数统计:")
print(f"  可训练参数: {trainable:,} ({trainable/total*100:.2f}%)")
print(f"  冻结参数:   {frozen:,} ({frozen/total*100:.2f}%)")
print(f"  总参数:     {total:,}")
expected_trainable = 8 * 64 + 128 * 8  # A: r×in + B: out×r
assert trainable == expected_trainable, f"可训练参数应为 {expected_trainable}"
print(f"  ✅ 通过：可训练参数 = r×in + out×r = {8}×{64} + {128}×{8} = {expected_trainable}")

Test 1 - 初始时 LoRA 输出 vs 基座输出最大差异: 0.00e+00
  ✅ 通过：B=0 保证了初始输出等于预训练模型

Test 2 - merge 前后输出最大差异: 9.54e-07
  ✅ 通过：权重合并后输出不变
Test 2 - unmerge 后输出最大差异: 2.38e-07
  ✅ 通过：unmerge 后输出恢复

Test 3 - 参数统计:
  可训练参数: 1,536 (15.79%)
  冻结参数:   8,192 (84.21%)
  总参数:     9,728
  ✅ 通过：可训练参数 = r×in + out×r = 8×64 + 128×8 = 1536


## Part 2：LoRA 注入 LLaMA 模型

### 2.1 构建缩小版 LLaMA 模型

复用 Week4 的 LLaMA 架构，使用缩小配置便于本地实验。

In [4]:
@dataclass
class LLaMAConfig:
    vocab_size: int = 1000
    d_model: int = 256
    n_heads: int = 8
    n_kv_heads: int = 4  # GQA
    n_layers: int = 4
    d_ff: int = 688  # ≈ 2/3 × 4 × d_model for SwiGLU
    max_seq_len: int = 256
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    dropout: float = 0.0


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * rms * self.weight


def precompute_rope_freqs(dim: int, max_seq_len: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len)
    freqs = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(freqs), freqs)  # complex64


def apply_rope(x: torch.Tensor, freqs: torch.Tensor) -> torch.Tensor:
    x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    freqs = freqs[:x.shape[-2], :].unsqueeze(0).unsqueeze(0)
    x_rotated = x_complex * freqs
    return torch.view_as_real(x_rotated).reshape(*x.shape).type_as(x)


class GQAttention(nn.Module):
    def __init__(self, config: LLaMAConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.d_head = config.d_model // config.n_heads
        self.n_rep = self.n_heads // self.n_kv_heads

        self.wq = nn.Linear(config.d_model, config.n_heads * self.d_head, bias=False)
        self.wk = nn.Linear(config.d_model, config.n_kv_heads * self.d_head, bias=False)
        self.wv = nn.Linear(config.d_model, config.n_kv_heads * self.d_head, bias=False)
        self.wo = nn.Linear(config.n_heads * self.d_head, config.d_model, bias=False)

    def forward(self, x, freqs, mask=None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        q = apply_rope(q, freqs)
        k = apply_rope(k, freqs)

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.wo(out)


class SwiGLUFFN(nn.Module):
    def __init__(self, config: LLaMAConfig):
        super().__init__()
        self.w_gate = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.w_up = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.w_down = nn.Linear(config.d_ff, config.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class LLaMABlock(nn.Module):
    def __init__(self, config: LLaMAConfig):
        super().__init__()
        self.attention = GQAttention(config)
        self.ffn = SwiGLUFFN(config)
        self.attention_norm = RMSNorm(config.d_model, config.norm_eps)
        self.ffn_norm = RMSNorm(config.d_model, config.norm_eps)

    def forward(self, x, freqs, mask=None):
        x = x + self.attention(self.attention_norm(x), freqs, mask)
        x = x + self.ffn(self.ffn_norm(x))
        return x


class LLaMAModel(nn.Module):
    def __init__(self, config: LLaMAConfig):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.layers = nn.ModuleList([LLaMABlock(config) for _ in range(config.n_layers)])
        self.norm = RMSNorm(config.d_model, config.norm_eps)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        self.freqs = precompute_rope_freqs(
            config.d_model // config.n_heads,
            config.max_seq_len,
            config.rope_theta,
        )

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        x = self.tok_emb(input_ids)
        freqs = self.freqs[:T].to(x.device)
        mask = torch.tril(torch.ones(T, T, device=x.device)).unsqueeze(0).unsqueeze(0)

        for layer in self.layers:
            x = layer(x, freqs, mask)

        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )
        return logits, loss


config = LLaMAConfig()
model = LLaMAModel(config)
total_params = sum(p.numel() for p in model.parameters())
print(f"LLaMA 模型总参数: {total_params:,}")

LLaMA 模型总参数: 3,414,272


### 2.2 自动注入 LoRA

核心函数：遍历模型的所有 `nn.Linear` 层，将匹配目标名称的层替换为 `LoRALinear`。

In [5]:
def inject_lora(
    model: nn.Module,
    target_modules: Set[str],
    r: int = 8,
    lora_alpha: int = 16,
    lora_dropout: float = 0.0,
) -> nn.Module:
    """
    将模型中匹配 target_modules 的 nn.Linear 替换为 LoRALinear。
    
    Args:
        model: 预训练模型
        target_modules: 要替换的模块名称集合（如 {'wq', 'wv'}）
        r: LoRA 秩
        lora_alpha: LoRA 缩放常数
        lora_dropout: LoRA dropout 率
    """
    for name, module in model.named_modules():
        # 检查该模块的子模块
        for child_name, child_module in module.named_children():
            if child_name in target_modules and isinstance(child_module, nn.Linear):
                # 创建 LoRALinear 替代
                lora_linear = LoRALinear(
                    in_features=child_module.in_features,
                    out_features=child_module.out_features,
                    r=r,
                    lora_alpha=lora_alpha,
                    lora_dropout=lora_dropout,
                    bias=child_module.bias is not None,
                )
                # 复制预训练权重
                lora_linear.linear.weight.data = child_module.weight.data.clone()
                if child_module.bias is not None:
                    lora_linear.linear.bias.data = child_module.bias.data.clone()

                setattr(module, child_name, lora_linear)

    return model


def freeze_non_lora_params(model: nn.Module):
    """冻结所有非 LoRA 参数。"""
    for name, param in model.named_parameters():
        if 'lora_' not in name:
            param.requires_grad = False


def print_trainable_parameters(model: nn.Module):
    """打印可训练参数统计信息。"""
    trainable = 0
    total = 0
    for name, param in model.named_parameters():
        total += param.numel()
        if param.requires_grad:
            trainable += param.numel()
    print(f"可训练参数: {trainable:,} / {total:,} ({trainable / total * 100:.4f}%)")
    return trainable, total

In [6]:
# 注入 LoRA 到 Attention 的 Q 和 V 投影
target_modules = {'wq', 'wv'}
lora_r = 8
lora_alpha = 16

lora_model = LLaMAModel(config)
lora_model = inject_lora(lora_model, target_modules, r=lora_r, lora_alpha=lora_alpha)
freeze_non_lora_params(lora_model)

print("=== LoRA 注入后的模型结构 (部分) ===")
for name, module in lora_model.named_modules():
    if isinstance(module, LoRALinear):
        print(f"  {name}: {module}")

print()
print_trainable_parameters(lora_model)

# 列出所有可训练参数
print("\n可训练参数列表:")
for name, param in lora_model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.shape}")

=== LoRA 注入后的模型结构 (部分) ===
  layers.0.attention.wq: LoRALinear(
  in_features=256, out_features=256, r=8, alpha=16, scaling=2.00, merged=False
  (linear): Linear(in_features=256, out_features=256, bias=False)
  (lora_dropout): Identity()
)
  layers.0.attention.wv: LoRALinear(
  in_features=256, out_features=128, r=8, alpha=16, scaling=2.00, merged=False
  (linear): Linear(in_features=256, out_features=128, bias=False)
  (lora_dropout): Identity()
)
  layers.1.attention.wq: LoRALinear(
  in_features=256, out_features=256, r=8, alpha=16, scaling=2.00, merged=False
  (linear): Linear(in_features=256, out_features=256, bias=False)
  (lora_dropout): Identity()
)
  layers.1.attention.wv: LoRALinear(
  in_features=256, out_features=128, r=8, alpha=16, scaling=2.00, merged=False
  (linear): Linear(in_features=256, out_features=128, bias=False)
  (lora_dropout): Identity()
)
  layers.2.attention.wq: LoRALinear(
  in_features=256, out_features=256, r=8, alpha=16, scaling=2.00, merged=False
  (li

### 2.3 验证注入正确性

注入 LoRA 后，模型的初始输出应与原始模型完全一致（因为 B=0）。

In [7]:
# 创建两个相同权重的模型
torch.manual_seed(123)
original_model = LLaMAModel(config)

torch.manual_seed(123)
lora_test_model = LLaMAModel(config)
lora_test_model = inject_lora(lora_test_model, target_modules, r=lora_r, lora_alpha=lora_alpha)
freeze_non_lora_params(lora_test_model)

# 测试输出一致性
test_input = torch.randint(0, config.vocab_size, (2, 32))
with torch.no_grad():
    orig_logits, _ = original_model(test_input)
    lora_logits, _ = lora_test_model(test_input)

diff = (orig_logits - lora_logits).abs().max().item()
print(f"原始模型 vs LoRA 模型（初始状态）输出最大差异: {diff:.2e}")
assert diff < 1e-5, "初始 LoRA 模型应与原始模型输出一致！"
print("✅ 通过：LoRA 注入不改变模型初始行为")

原始模型 vs LoRA 模型（初始状态）输出最大差异: 0.00e+00
✅ 通过：LoRA 注入不改变模型初始行为


## Part 3：权重合并与 LoRA 权重管理

### 3.1 全模型 merge / unmerge

In [8]:
def merge_lora_weights(model: nn.Module):
    """合并模型中所有 LoRA 权重。"""
    for module in model.modules():
        if isinstance(module, LoRALinear):
            module.merge()


def unmerge_lora_weights(model: nn.Module):
    """从模型中移除所有 LoRA 合并。"""
    for module in model.modules():
        if isinstance(module, LoRALinear):
            module.unmerge()


def save_lora_weights(model: nn.Module, path: str):
    """只保存 LoRA 权重（A 和 B 矩阵）。"""
    lora_state = {}
    for name, param in model.named_parameters():
        if 'lora_' in name:
            lora_state[name] = param.data.clone()
    torch.save(lora_state, path)
    size_mb = sum(v.numel() * v.element_size() for v in lora_state.values()) / 1024 / 1024
    print(f"LoRA 权重已保存到 {path}，大小: {size_mb:.2f} MB")


def load_lora_weights(model: nn.Module, path: str):
    """加载 LoRA 权重。"""
    lora_state = torch.load(path, weights_only=True)
    model_state = model.state_dict()
    for name, param in lora_state.items():
        if name in model_state:
            model_state[name].copy_(param)
    print(f"LoRA 权重已从 {path} 加载")

### 3.2 LoRA 权重大小对比

对比保存完整模型 vs 只保存 LoRA 权重的存储差异。

In [9]:
import tempfile
import os

with tempfile.TemporaryDirectory() as tmpdir:
    # 保存完整模型
    full_path = os.path.join(tmpdir, 'full_model.pt')
    torch.save(lora_model.state_dict(), full_path)
    full_size = os.path.getsize(full_path) / 1024 / 1024

    # 只保存 LoRA 权重
    lora_path = os.path.join(tmpdir, 'lora_weights.pt')
    save_lora_weights(lora_model, lora_path)
    lora_size = os.path.getsize(lora_path) / 1024 / 1024

    print(f"\n存储对比:")
    print(f"  完整模型: {full_size:.2f} MB")
    print(f"  LoRA 权重: {lora_size:.2f} MB")
    print(f"  压缩比: {full_size / lora_size:.1f}x")

LoRA 权重已保存到 /tmp/tmpxpwh18v_/lora_weights.pt，大小: 0.11 MB

存储对比:
  完整模型: 13.15 MB
  LoRA 权重: 0.12 MB
  压缩比: 114.4x


## Part 4：LoRA SFT 训练验证

### 4.1 准备 SFT 训练数据

复用 Week5 Day6 的 SFT 数据处理流程。

In [10]:
# 简易 Tokenizer
class SimpleTokenizer:
    def __init__(self, vocab_size=1000):
        self.vocab_size = vocab_size
        self.char_to_id = {}
        self.id_to_char = {}
        # 特殊 token
        self.pad_token_id = 0
        self.bos_token_id = 1
        self.eos_token_id = 2
        self.unk_token_id = 3

        specials = ['<pad>', '<bos>', '<eos>', '<unk>']
        for i, s in enumerate(specials):
            self.char_to_id[s] = i
            self.id_to_char[i] = s

        chars = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,!?;:'-\n")
        for i, c in enumerate(chars):
            self.char_to_id[c] = i + len(specials)
            self.id_to_char[i + len(specials)] = c

    def encode(self, text, max_len=None):
        ids = [self.char_to_id.get(c, self.unk_token_id) for c in text]
        if max_len:
            ids = ids[:max_len]
        return ids

    def decode(self, ids):
        return ''.join(self.id_to_char.get(i, '?') for i in ids
                       if i not in [self.pad_token_id, self.bos_token_id, self.eos_token_id])


tokenizer = SimpleTokenizer(vocab_size=config.vocab_size)

In [ ]:
# SFT 数据
SFT_DATA = [
    {"instruction": "What is AI?", "input": "", "output": "AI is artificial intelligence."},
    {"instruction": "Translate to French.", "input": "Hello world", "output": "Bonjour le monde."},
    {"instruction": "Summarize the text.", "input": "Machine learning is a subset of AI.", "output": "ML is part of AI."},
    {"instruction": "What is Python?", "input": "", "output": "Python is a programming language."},
    {"instruction": "Explain briefly.", "input": "Neural networks mimic the brain.", "output": "NN copies brain structure."},
    {"instruction": "Capital of Japan?", "input": "", "output": "Tokyo is the capital of Japan."},
    {"instruction": "Define LoRA.", "input": "", "output": "LoRA is low rank adaptation."},
    {"instruction": "What is fine tuning?", "input": "", "output": "Adapting a model to a task."},
]

PROMPT_TEMPLATE = """### Instruction:\n{instruction}\n### Input:\n{input}\n### Response:\n{output}"""


class SFTDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=128):
        self.samples = []
        for item in data:
            prompt_part = f"### Instruction:\n{item['instruction']}\n### Input:\n{item['input']}\n### Response:\n"
            full_text = prompt_part + item['output']

            input_ids = tokenizer.encode(full_text, max_len=max_len)
            prompt_len = len(tokenizer.encode(prompt_part, max_len=max_len))

            labels = [-100] * prompt_len + input_ids[prompt_len:]

            # Padding
            pad_len = max_len - len(input_ids)
            input_ids = input_ids + [tokenizer.pad_token_id] * pad_len
            labels = labels + [-100] * pad_len

            self.samples.append({
                'input_ids': torch.tensor(input_ids, dtype=torch.long),
                'labels': torch.tensor(labels, dtype=torch.long),
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


dataset = SFTDataset(SFT_DATA, tokenizer, max_len=128)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
print(f"数据集大小: {len(dataset)} 条")
print(f"样本 input_ids shape: {dataset[0]['input_ids'].shape}")

### 4.2 训练循环

对比 Full FT 和 LoRA 的训练过程。

In [ ]:
def train_model(model, dataloader, epochs=20, lr=1e-3, label=""):
    """通用训练循环。"""
    model = model.to(device)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
    )
    model.train()
    losses = []

    t0 = time.time()
    for epoch in range(epochs):
        epoch_loss = 0
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            _, loss = model(input_ids, labels=labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(dataloader)
        losses.append(avg_loss)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  [{label}] Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    elapsed = time.time() - t0
    print(f"  [{label}] 训练完成，耗时: {elapsed:.1f}s，最终 Loss: {losses[-1]:.4f}")
    return losses

In [ ]:
# ===== Full FT 训练 =====
print("=" * 60)
print("Full Fine-tuning")
print("=" * 60)

torch.manual_seed(42)
full_ft_model = LLaMAModel(config)
full_ft_trainable, full_ft_total = print_trainable_parameters(full_ft_model)
full_ft_losses = train_model(full_ft_model, dataloader, epochs=30, lr=1e-3, label="Full FT")

In [ ]:
# ===== LoRA 训练 =====
print("\n" + "=" * 60)
print("LoRA Fine-tuning (Wq + Wv, r=8)")
print("=" * 60)

torch.manual_seed(42)
lora_ft_model = LLaMAModel(config)
lora_ft_model = inject_lora(lora_ft_model, {'wq', 'wv'}, r=8, lora_alpha=16)
freeze_non_lora_params(lora_ft_model)
lora_trainable, lora_total = print_trainable_parameters(lora_ft_model)
lora_ft_losses = train_model(lora_ft_model, dataloader, epochs=30, lr=1e-3, label="LoRA")

In [ ]:
# ===== LoRA 全部 Linear 训练 =====
print("\n" + "=" * 60)
print("LoRA Fine-tuning (All Linear, r=8)")
print("=" * 60)

torch.manual_seed(42)
lora_all_model = LLaMAModel(config)
all_targets = {'wq', 'wk', 'wv', 'wo', 'w_gate', 'w_up', 'w_down'}
lora_all_model = inject_lora(lora_all_model, all_targets, r=8, lora_alpha=16)
freeze_non_lora_params(lora_all_model)
lora_all_trainable, lora_all_total = print_trainable_parameters(lora_all_model)
lora_all_losses = train_model(lora_all_model, dataloader, epochs=30, lr=1e-3, label="LoRA-All")

### 4.3 训练结果对比

In [ ]:
print("\n" + "=" * 70)
print("训练结果对比")
print("=" * 70)
print(f"{'方法':<25} {'可训练参数':>12} {'占比':>8} {'最终Loss':>10}")
print("-" * 70)
print(f"{'Full FT':<25} {full_ft_trainable:>12,} {'100%':>8} {full_ft_losses[-1]:>10.4f}")
print(f"{'LoRA (Wq+Wv, r=8)':<25} {lora_trainable:>12,} {lora_trainable/lora_total*100:>7.2f}% {lora_ft_losses[-1]:>10.4f}")
print(f"{'LoRA (All Linear, r=8)':<25} {lora_all_trainable:>12,} {lora_all_trainable/lora_all_total*100:>7.2f}% {lora_all_losses[-1]:>10.4f}")

### 4.4 文本生成对比

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=50, temperature=0.8):
    model.eval()
    model = model.to(device)
    input_ids = torch.tensor([tokenizer.encode(prompt)], device=device)

    for _ in range(max_new_tokens):
        if input_ids.shape[1] >= config.max_seq_len:
            break
        logits, _ = model(input_ids)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_token], dim=-1)
        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(input_ids[0].tolist())


test_prompt = "### Instruction:\nWhat is LoRA?\n### Input:\n\n### Response:\n"

print("=== 文本生成对比 ===")
print(f"\nPrompt: {test_prompt.strip()}")
print(f"\n[Full FT]:     {generate(full_ft_model, tokenizer, test_prompt)}")

# 合并 LoRA 权重后再生成
merge_lora_weights(lora_ft_model)
print(f"[LoRA Wq+Wv]: {generate(lora_ft_model, tokenizer, test_prompt)}")
unmerge_lora_weights(lora_ft_model)

merge_lora_weights(lora_all_model)
print(f"[LoRA All]:    {generate(lora_all_model, tokenizer, test_prompt)}")
unmerge_lora_weights(lora_all_model)

## Part 5：参数效率分析

### 5.1 不同 rank 的参数量对比

In [ ]:
print("=" * 70)
print("不同 LoRA 配置的参数效率分析 (LLaMA-7B 参数: d=4096, L=32)")
print("=" * 70)

d = 4096
L = 32
total_7b = 6_738_000_000

configs = [
    ("Wq + Wv", 2, 4),
    ("Wq + Wv", 2, 8),
    ("Wq + Wv", 2, 16),
    ("Wq + Wv", 2, 64),
    ("All Attn (Wq,Wk,Wv,Wo)", 4, 8),
    ("All Attn (Wq,Wk,Wv,Wo)", 4, 16),
    ("All Linear (7 matrices)", 7, 8),
    ("All Linear (7 matrices)", 7, 16),
]

print(f"{'Target Modules':<30} {'n_mat':>5} {'rank':>5} {'LoRA Params':>14} {'占比':>8}")
print("-" * 70)
for name, n_mat, r in configs:
    lora_params = L * n_mat * 2 * r * d
    ratio = lora_params / total_7b * 100
    print(f"{name:<30} {n_mat:>5} {r:>5} {lora_params:>14,} {ratio:>7.3f}%")

In [ ]:
print("\n" + "=" * 70)
print("显存估算对比 (LLaMA-7B, FP16)")
print("=" * 70)

methods = [
    ("Full FT (FP16)", 6.7e9 * 2, 6.7e9 * 2 * 4, 6.7e9 * 2, 10),
    ("LoRA (r=16, Wq+Wv)", 6.7e9 * 2, 8.4e6 * 4, 8.4e6 * 2, 8),
    ("LoRA (r=16, All)", 6.7e9 * 2, 25.2e6 * 4, 25.2e6 * 2, 8),
    ("QLoRA (r=16, All)", 6.7e9 * 0.5, 25.2e6 * 4, 25.2e6 * 2, 6),
]

print(f"{'方法':<25} {'模型(GB)':>9} {'优化器(GB)':>11} {'梯度(GB)':>9} {'激活(GB)':>9} {'总计(GB)':>9}")
print("-" * 80)
for name, model_mem, opt_mem, grad_mem, act_mem in methods:
    model_gb = model_mem / 1e9
    opt_gb = opt_mem / 1e9
    grad_gb = grad_mem / 1e9
    total_gb = model_gb + opt_gb + grad_gb + act_mem
    print(f"{name:<25} {model_gb:>8.1f} {opt_gb:>10.2f} {grad_gb:>8.2f} {act_mem:>8.0f} {total_gb:>8.1f}")

## 总结

本日实现了 LoRA 的完整代码：

1. **LoRALinear 层**：核心前向传播 $h = W_0 x + \frac{\alpha}{r} BAx$，含初始化、Dropout、merge/unmerge
2. **LoRA 注入**：自动替换指定的 Linear 层，冻结非 LoRA 参数
3. **权重管理**：合并、卸载、保存、加载 LoRA 权重
4. **训练验证**：Full FT vs LoRA 的效果和效率对比

**面试高频手写**：
- LoRALinear 的 `forward` 方法
- `merge` / `unmerge` 的实现
- LoRA 参数量计算

**明日预告**：Day 4 将深入 QLoRA 的量化原理——NF4、双重量化、分页优化器。